In [1]:
import os
import sys

# Third-party numerical and data handling
import numpy as np
import pandas as pd
# Visualization
import matplotlib.pyplot as plt

# PyTorch core
import torch
from torch import nn
from torch.utils.data import DataLoader

from model_evaluation_helpers import check_device, read_preprocessed_images, ECGDataset, val_transforms, MultiHeadEfficientNet, get_probs_and_labels, compute_ranking_metrics
from sklearn.model_selection import train_test_split

In [2]:
# Check and get device
device = check_device()

try: 
    print(image_set["train_000000.png"])
except Exception as e:
    image_set = read_preprocessed_images("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/proc_images.h5")
    

label_df = pd.read_csv("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/train_final.csv", index_col=0)

image_names = list(image_set.keys())
image_ids = list([int(image_name.split(".")[0][-6:]) for image_name in image_names])
label_df = label_df.loc[label_df.index.isin(set(image_ids))]

# need to order label_df so that it has the same ordering as image_names
image_df = pd.DataFrame({
    "image_name": image_set.keys(),
})
image_df["image_id"] = image_df["image_name"].str.split(".", expand=True)[0].str[-6:].astype(int)
image_df = image_df.sort_values(by="image_id")
image_df = image_df.set_index("image_id")
# ensure ordering of label_df and image_df
label_df = label_df.loc[image_df.index]
X_train, X_test, y_train, y_test = train_test_split(image_df,
                                                    label_df,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    shuffle=True,
                                                    stratify=label_df[["CD", "MI", "AF", "STTC", "HYP"]]
                                                    )

train_image_names = X_train["image_name"].tolist()
test_image_names = X_test["image_name"].tolist()

val_dataset = ECGDataset(
    image_names = test_image_names,
    image_set = image_set,
    labels_df = y_test,
    transforms = val_transforms
)

num_workers = 0 if sys.platform == 'darwin' else 4 
print(f"Using num_workers = {num_workers}")

val_dataloader = DataLoader( 
                        val_dataset,
                        batch_size=32, 
                        shuffle=False, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)

✓ MPS (Apple Silicon GPU) available

Selected device: mps
Using num_workers = 0


In [3]:
def evaluate_model(modelpath, modeltype, verbose=True, model_size=512):
    checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
    current_epoch = checkpoint["epoch"]
    if verbose:
        print(f"Loading from checkpoint, last run epoch was {current_epoch}")
        
    model = MultiHeadEfficientNet(
        num_conditions=5, 
        hidden_dim=model_size, 
        dropout_rate=0.0, 
        model=modeltype
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
    ranking_metrics = compute_ranking_metrics(val_labels, val_probs)
    
    return ranking_metrics

In [4]:
def evaluate_multiple_models(modeldict, modelsizes, modeltype="convnext"):
    """
    Takes in dictionary with format {str: str}, representing {modelname: path to model checkpoint} and collects ranking metrics
    for those models
    """
    collected_results = {}
    for modelname, modelpath in modeldict.items(): 
        results = evaluate_model(modelpath, modeltype, model_size=modelsizes[modelname])
        collected_results[modelname] = results
    return collected_results

In [6]:
basepath = os.path.join(os.path.expanduser("~"), "Library", "CloudStorage", "OneDrive-Nexus365", "BHF_Cardiac_Problem", "Results_Collection")

modeldict = {
    #"BCE_Weighted": os.path.join(basepath, "Model_BCE_Weighted", "best_model.pth"), 
    "LR_Tuned": os.path.join(basepath, "Model_LR_Tuned", "best_model.pth")
}
modelsizes = {
    #"BCE_Weighted": 512, 
    "LR_Tuned": 750
}

agg_results = evaluate_multiple_models(modeldict, modelsizes)

Loading from checkpoint, last run epoch was 15


In [7]:
from tabulate import tabulate
models = agg_results.keys()
to_write = []
for model in models: 
    to_write.append([model, 
                    agg_results[model]["macro_f1"],
                    agg_results[model]["macro_precision"], 
                    agg_results[model]["macro_recall"], 
                    agg_results[model]["macro_ece"], 
                    agg_results[model]["micro_ece"],
                    agg_results[model]["macro_brier"], 
                    agg_results[model]["macro_auroc"], 
                    agg_results[model]["micro_auroc"], 
                    agg_results[model]["macro_ap"], 
                    agg_results[model]["micro_ap"]])
    
print(tabulate(to_write, headers=["Model Name", "Macro F1", "Macro_Precision", "Macro_Recall", "Macro_ECE", "Micro_ECE", "Macro_Brier", "Macro_AUROC", "Micro_AUROC", "Macro_AP", "Micro_AP"]))

Model Name      Macro F1    Macro_Precision    Macro_Recall    Macro_ECE    Micro_ECE    Macro_Brier    Macro_AUROC    Micro_AUROC    Macro_AP    Micro_AP
------------  ----------  -----------------  --------------  -----------  -----------  -------------  -------------  -------------  ----------  ----------
LR_Tuned         0.77557           0.769847        0.783336    0.0204256    0.0143015      0.0618614       0.941408       0.949052    0.836824    0.841378


In [8]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "f1"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_f1    HYP_f1     MI_f1     CD_f1     AF_f1
------------  ---------  --------  --------  --------  --------
LR_Tuned       0.760728  0.663172  0.770819  0.797417  0.885714


In [9]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "precision"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_precision    HYP_precision    MI_precision    CD_precision    AF_precision
------------  ----------------  ---------------  --------------  --------------  --------------
LR_Tuned              0.721332         0.643766         0.81042         0.80456        0.869159


In [10]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "recall"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_recall    HYP_recall    MI_recall    CD_recall    AF_recall
------------  -------------  ------------  -----------  -----------  -----------
LR_Tuned           0.804677      0.683784     0.734908       0.7904     0.902913


In [11]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "specificity"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_specificity    HYP_specificity    MI_specificity    CD_specificity    AF_specificity
------------  ------------------  -----------------  ----------------  ----------------  ----------------
LR_Tuned                0.900659           0.946809          0.941518          0.949516          0.989986


In [12]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "auroc"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_auroc    HYP_auroc    MI_auroc    CD_auroc    AF_auroc
------------  ------------  -----------  ----------  ----------  ----------
LR_Tuned          0.936865     0.918467    0.929557     0.93828    0.983869


In [16]:
agg_results

{'LR_Tuned': {'per_label_ap': [0.8294650922718465,
   0.7217309664811656,
   0.846648963065704,
   0.8740106270148053,
   0.9122654880371459],
  'per_label_auroc': [0.9368653354898198,
   0.9184670993181632,
   0.9295568522684664,
   0.9382798485485907,
   0.9838691195466478],
  'per_label_brier': [0.08520891968551383,
   0.0607697585294146,
   0.08672052978198944,
   0.06298696054789478,
   0.013620984847766391],
  'per_label_brier_baseline': [0.18352466821670532,
   0.1080603152513504,
   0.18940071761608124,
   0.16484956443309784,
   0.06391208618879318],
  'relative_brier_improvement': [0.5357086297254633,
   0.4376311193608592,
   0.5421319893952379,
   0.6179124842428245,
   0.7868793578802816],
  'per_label_f1': [0.7607282184655396,
   0.6631716906946264,
   0.7708189951823813,
   0.7974172719935432,
   0.8857142857142857],
  'per_label_precision': [0.7213316892725031,
   0.6437659033078881,
   0.8104196816208393,
   0.8045602605863192,
   0.8691588785046729],
  'per_label_reca

In [24]:
models = agg_results.keys()
conditions = ["STTC", "HYP", "MI", "CD", "AF"]
metrics = ["auroc", "f1", "specificity", "recall"]

results = []
for metric in metrics: 
    res = agg_results["LR_Tuned"][f"per_label_{metric}"]
    results.append(res)
df = pd.DataFrame(results, columns=conditions, index=metrics)
df = df.rename(index = {"recall": "sensitivity"})

In [26]:
print(df)

                 STTC       HYP        MI        CD        AF
auroc        0.936865  0.918467  0.929557  0.938280  0.983869
f1           0.760728  0.663172  0.770819  0.797417  0.885714
specificity  0.900659  0.946809  0.941518  0.949516  0.989986
sensitivity  0.804677  0.683784  0.734908  0.790400  0.902913
